# Real-World Excel Formula Extraction (Enhanced)

This notebook demonstrates how the backend processes `Employee_Payroll_Final.xlsx` by extracting **both** the rich statistical metadata (data types, averages) **and** the raw Excel formulas.

We will:
1. Load `Employee_Payroll_Final.xlsx` using `pandas` to get data types and statistics.
2. Load it again using `openpyxl` to extract the raw Excel formulas.
3. Combine both into a master JSON metadata payload.
4. Ask `gpt-4.1-mini` to translate those Excel formulas into vectorized `pandas` Python code.

In [21]:
import pandas as pd
import openpyxl
import json
import os
import re

file_path = "Enterprise_Payroll_Realistic.xlsx"

if not os.path.exists(file_path):
    print(f"Error: {file_path} not found.")
else:
    print(f"Successfully found {file_path}. Proceeding...")

Successfully found Enterprise_Payroll_Realistic.xlsx. Proceeding...


### Step 1: Scanning the Excel File for Data Types, Statistics, AND Formulas
We use `pandas` to grab the column types and summary stats, and `openpyxl` to find the `=SUM(...)` formula strings. We weave them together into one beautiful JSON block for the AI.

In [33]:
# 1a. Load all sheets into a dictionary of DataFrames
all_sheets = pd.read_excel(file_path, sheet_name=None)
wb = openpyxl.load_workbook(file_path, data_only=False)

global_metadata = {"sheets": {}}

# 1b. Iterate through every sheet in the workbook
for sheet_name in wb.sheetnames:
    df_raw = all_sheets[sheet_name]
    sheet = wb[sheet_name]
    
    headers = [cell.value for cell in sheet[1]]
    
    # Create mappings for the LLM
    from openpyxl.utils import get_column_letter
    col_map = {get_column_letter(idx + 1): header for idx, header in enumerate(headers) if header}
    
    # CRITICAL: Excel VLOOKUPS use 1-based indexing (e.g., "get column 7"). 
    # We must map these numbers to header names for the AI.
    vlookup_index_map = {str(idx + 1): header for idx, header in enumerate(headers) if header}
    
    # Extract formulas just for this sheet
    detected_formulas = {}
    for col_idx, cell in enumerate(sheet[2], start=1):
        if col_idx <= len(headers) and headers[col_idx - 1]:
            header_name = headers[col_idx - 1]
            if cell.value and str(cell.value).startswith("="):
                cell_val = str(cell.value)
                
                # Regex to replace local cell references (like A2) with [Header_Name]
                def replace_local(match):
                    col_letter = match.group(1)
                    return f"[{col_map.get(col_letter, col_letter)}]"
                
                safe_formula = re.sub(r'([A-Z]{1,3})\d+', replace_local, cell_val)
                detected_formulas[header_name] = safe_formula
                
    # Build the massive JSON context
    global_metadata["sheets"][sheet_name] = {
        "columns": {col: str(dtype) for col, dtype in df_raw.dtypes.items()},
        "vlookup_column_indices": vlookup_index_map,
        "formulas_detected": detected_formulas
    }

metadata_json = json.dumps(global_metadata, indent=2)
print(metadata_json)

print("\n--- MASTER MULTI-SHEET JSON PAYLOAD ---")
print(json.dumps(global_metadata['sheets']['Payroll_Run']['formulas_detected'], indent=2))

{
  "sheets": {
    "Employees": {
      "columns": {
        "Employee_ID": "object",
        "First_Name": "object",
        "Last_Name": "object",
        "Email": "object",
        "Location": "object",
        "Department": "object",
        "Role": "object",
        "Join_Date": "object",
        "Base_Salary": "int64"
      },
      "vlookup_column_indices": {
        "1": "Employee_ID",
        "2": "First_Name",
        "3": "Last_Name",
        "4": "Email",
        "5": "Location",
        "6": "Department",
        "7": "Role",
        "8": "Join_Date",
        "9": "Base_Salary"
      },
      "formulas_detected": {}
    },
    "Benefits": {
      "columns": {
        "Role": "object",
        "Health_Ins_Cost": "int64",
        "PF_Percentage": "float64"
      },
      "vlookup_column_indices": {
        "1": "Role",
        "2": "Health_Ins_Cost",
        "3": "PF_Percentage"
      },
      "formulas_detected": {}
    },
    "Tax_Brackets": {
      "columns": {
        "

### Step 2: Auto-Translate Excel Logic to Python!
Now we give this rich JSON context to our AI Data Scientist. Because it now knows `Role` is an `object` (string) and `Base_Salary` is an `int64`, it will write even better Python code!

In [ ]:
import dspy
from dotenv import load_dotenv
import sys

# Setup DSPy
load_dotenv()
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils.model_registry import get_model_object

dspy.settings.configure(lm=get_model_object("gpt-4.1-mini"))

prompt = f"""
You are an Expert AI Data Engineer. Your goal is to translate mapped Excel formulas from a multi-sheet workbook into vectorized Python Pandas code.

Here is the global schema, including data types, VLOOKUP index maps, and the formulas extracted from the Excel file:
{metadata_json}

User Query: Write a Python script that calculates ALL the missing formulas in the 'Payroll_Run' sheet using Pandas.

CRITICAL RULES:
1. Assume the data is loaded in a dictionary of DataFrames called `all_sheets`.
2. Write ONLY standard Python code. NO markdown formatting, NO explanations, NO ```python blocks.
3. RELATIONAL LOGIC (VLOOKUP): 
   - EXACT match VLOOKUP (FALSE) -> use `pd.merge()`.
   - APPROXIMATE match VLOOKUP (TRUE) -> use `pd.merge_asof()`. 
   - For Tax Brackets, you MUST merge on `Min_Income` (not Max_Income) using `direction='backward'`.
   - CRITICAL PANDAS RULE: Cast both the `left_on` and `right_on` columns to `float64` before using `pd.merge_asof()`.
4. PLACEHOLDER COLUMNS: Do NOT create duplicate columns. Assign the merged values directly to the existing `_F` columns and drop the extra columns.
5. Order of Operations: Calculate dependent columns in the correct logical order.
"""

print("\nRequesting translation from OpenAI (gpt-4.1-mini)...")

try:
    lm = get_model_object("gpt-4.1-mini")
    response = lm(prompt)
    code = response[0] if isinstance(response, list) else response
    
    code = code.strip()
    if code.startswith("```"):
        code = "\n".join(code.split("\n")[1:-1])
        
    print("\n--- AI TRANSLATED PYTHON CODE ---")
    print(code)
except Exception as e:
    print(f"Error during LLM translation: {e}")


Requesting translation from OpenAI (gpt-4.1-mini)...

--- AI TRANSLATED PYTHON CODE ---
all_sheets['Payroll_Run'] = all_sheets['Payroll_Run'].copy()

# Merge for Role_F and Base_Salary_F
payroll_run_merged = pd.merge(all_sheets['Payroll_Run'], all_sheets['Employees'], 
                               left_on='Employee_ID', right_on='Employee_ID', 
                               how='left', suffixes=('', '_emp'))

# Assigning the values to the existing columns
all_sheets['Payroll_Run']['Role_F'] = payroll_run_merged['Role']
all_sheets['Payroll_Run']['Base_Salary_F'] = payroll_run_merged['Base_Salary']

# Calculate Gross_Salary_F
all_sheets['Payroll_Run']['Gross_Salary_F'] = (
    all_sheets['Payroll_Run']['Base_Salary_F'] + 
    (all_sheets['Payroll_Run']['Base_Salary_F'] * (all_sheets['Payroll_Run']['Perf_Score'] / 5) * 0.1) + 
    (all_sheets['Payroll_Run']['OT_Hours'] * 45)
)

# Merge for Health_Deduct_F
payroll_run_merged_benefits = pd.merge(all_sheets['Payroll_Run'], all_sheets['Be

### Step 3: Execute the AI's Translated Code
Now we load the flat data in Pandas, and run the AI's translated Python script to dynamically calculate the missing columns purely in Python!

In [31]:
print("\n--- EXECUTING AI PYTHON SCRIPT LOCALLY ---")

all_sheets = pd.read_excel(file_path, sheet_name=None)

# 🔥 DOUBLE BULLETPROOF: Force both lookup columns to float
all_sheets['Tax_Brackets']['Min_Income'] = all_sheets['Tax_Brackets']['Min_Income'].astype(float)
all_sheets['Tax_Brackets']['Max_Income'] = all_sheets['Tax_Brackets']['Max_Income'].astype(float)

local_env = {
    "all_sheets": all_sheets, 
    "pd": pd,
    "__builtins__": {} 
}

try:
    exec(code, {}, local_env)
    
    # We make it global so Step 4 can safely access it!
    global df_payroll_final
    df_payroll_final = local_env["all_sheets"]["Payroll_Run"]
    
    print("\n✅ Execution Successful!")
    print("\n--- FINAL CALCULATED PAYROLL (First 5 Rows) ---")
    print(df_payroll_final[['Employee_ID', 'Gross_Salary_F', 'Tax_Rate_F', 'Net_Salary_F']].head())

except Exception as e:
    print("\n❌ Execution Error! The AI generated invalid code.")
    print(f"Details: {e}")


--- EXECUTING AI PYTHON SCRIPT LOCALLY ---

✅ Execution Successful!

--- FINAL CALCULATED PAYROLL (First 5 Rows) ---
  Employee_ID  Gross_Salary_F  Tax_Rate_F  Net_Salary_F
0     EMP1000      235220.308        0.12  206193.87104
1     EMP1001      165750.984        0.12  145360.86592
2     EMP1002      245921.298        0.12  215610.74224
3     EMP1003       91358.300        0.12   80145.30400
4     EMP1004      127677.680        0.12  112006.35840


### Step 4: Interact with the Computed Data!

Now that the AI has perfectly recalculated the Excel formulas into `df_final`, we can ask ANY question about the newly-calculated data (like Net Salary, Bonuses, or Hours)!


In [32]:
if 'df_payroll_final' not in globals():
    print("Error: Step 3 must complete successfully before running Step 4.")
else:
    user_question = "If we give everyone a 10% raise on their 'Gross_Salary_F', what will be the new total sum of 'Net_Salary_F' for the entire company? (Remember to recalculate the Tax_Rate_F based on the new Gross_Salary_F using the Tax_Brackets sheet)."

    print(f"Query: {user_question}")
    print("Asking AI for the python execution code...")

    # Pass the actual column headers into the prompt so it knows the exact schema!
    tax_cols = list(all_sheets['Tax_Brackets'].columns)
    payroll_cols = list(df_payroll_final.columns)

    prompt2 = f"""
    You are an AI Data Analyst.
    The user has a loaded dataframe `df_payroll_final` with columns: {payroll_cols}
    And a reference dataframe `df_tax` with columns: {tax_cols}

    User Question: {user_question}

    Rules:
    1. Write ONLY python code (no markdown quotes).
    2. To find the new tax rate, use `pd.merge_asof` on the `Min_Income` column with `direction='backward'`.
    3. Make sure to cast merge keys to float before merging!
    4. Save your final computed answer to a numeric variable called `answer`.
    """

    try:
        response2 = lm(prompt2)
        code2 = response2[0] if isinstance(response2, list) else response2
        code2 = code2.replace("```python", "").replace("```", "").strip()
        
        print("\n--- GENERATED CODE ---\n")
        print(code2)
        
        exec_env = {
            "df_payroll_final": df_payroll_final.copy(), 
            "df_tax": all_sheets['Tax_Brackets'].copy(),
            "pd": pd
        }
        exec(code2, {}, exec_env)
        
        print("\n--- FINAL ANSWER ---")
        print(f"${exec_env.get('answer', 0):,.2f}")
    except Exception as e:
        print(f"Error: {e}")

Query: If we give everyone a 10% raise on their 'Gross_Salary_F', what will be the new total sum of 'Net_Salary_F' for the entire company? (Remember to recalculate the Tax_Rate_F based on the new Gross_Salary_F using the Tax_Brackets sheet).
Asking AI for the python execution code...

--- GENERATED CODE ---

df_payroll_final['New_Gross_Salary_F'] = df_payroll_final['Gross_Salary_F'] * 1.10

# Merge keys to float for tax calculation
df_tax['Min_Income'] = df_tax['Min_Income'].astype(float)
df_tax['Max_Income'] = df_tax['Max_Income'].astype(float)
df_payroll_final['New_Gross_Salary_F'] = df_payroll_final['New_Gross_Salary_F'].astype(float)

# Determine the new Tax_Rate_F based on the new Gross_Salary_F
df_merged = pd.merge_asof(df_payroll_final.sort_values('New_Gross_Salary_F'), 
                          df_tax.sort_values('Min_Income'), 
                          left_on='New_Gross_Salary_F', 
                          right_on='Min_Income', 
                          direction='backwa